# CGT-02 预训练 (Pretraining) 教案

**课程名称：** 自建 GPT 预训练实战：从随机初始化到会说话的模型

**预计总时长：** 80-90 分钟

**源文件：** `Custom_GPT_Training/02_Pretraining.ipynb`（共 26 个 Cell，Cell 0-25）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 预训练概览 + 环境准备 | Cell 0-4 | 8 min |
| 8-20 min | 数据准备：语料加载与扩展 | Cell 5-6 | 12 min |
| 20-35 min | 数据集构建：滑动窗口 + 分词器 + DataLoader | Cell 7-10 | 15 min |
| 35-40 min | **休息 + 回顾** | -- | 5 min |
| 40-55 min | Trainer 实现：LR 调度 + 梯度裁剪 + 评估 | Cell 11-12 | 15 min |
| 55-70 min | 执行预训练 + 可视化分析 | Cell 13-17 | 15 min |
| 70-75 min | **休息 + 回顾** | -- | 5 min |
| 75-85 min | 模型测试：生成 + 温度实验 + 困惑度 | Cell 18-24 | 10 min |
| 85-90 min | 总结 + 下一步展望 | Cell 25 | 5 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 和 PyTorch 已安装
- [ ] 确认 `Custom_GPT_Training/custom_gpt.py` 模块可导入（Part 1 的产出）
- [ ] 确认 `data/custom_pretrain_corpus.txt` 存在
- [ ] 确认 `models/custom_gpt/` 目录可写
- [ ] 如有 GPU，确认 CUDA 可用
- [ ] 预跑一遍全部 Cell，确认 10 个 epoch 在 GPU 上约 1-2 分钟可完成
- [ ] 准备白板，用于手绘滑动窗口和 Next-Token Prediction 示意图

---

## 第一段：开场 + 预训练概览 + 环境准备（Cell 0-4）

📍 Cell 范围：Cell 0-1（Markdown 导读 + 流程表）、Cell 2（预训练关键机制）、Cell 3-4（环境设置代码）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：Part 1 组装了零件，现在要让模型学会说话
- 理解预训练 vs SFT vs DPO 的三阶段定位
- 确认环境就绪

🗣 讲课话术

> 大家好！上一节课我们组装了 CustomGPT 模型——词表、Embedding、Transformer Block、LM Head 全都有了。但现在这个模型就像一个刚出厂的婴儿，什么都不会。今天我们要做的就是**教它说话**。
>
> 打个比方：Part 1 是造了一辆车，Part 2 是教它学开。怎么教？很简单——给它看大量的文本，让它反复练习**预测下一个字**。
>
> 大家看 Cell 0 的示意图。输入序列 `[The, quick, brown, fox, jumps]`，模型要预测 `[quick, brown, fox, jumps, over]`——每个位置都在猜下一个词。这就是 Next-Token Prediction，预训练的全部核心。
>
> 再看这个三阶段对比表：
> - **预训练**：学语言规律，用大量原始文本，结果是「会说话」
> - **SFT**：学遵循指令，用 (指令, 回答) 对，结果是「会听话」
> - **DPO**：学人类偏好，用 (好回答, 差回答) 对，结果是「会选择」
>
> 今天我们只做第一步。运行 Cell 4。（运行 Cell 4）看到 `使用设备: cuda`，PyTorch 已就绪。CPU 也没问题，我们的模型不大。

👀 输出要点
- Cell 0：Next-Token Prediction 示意图 + 三阶段对比表（预训练/SFT/DPO）
- Cell 1：导航表（Part 1 -> **Part 2** -> Part 3）+ 四个学习目标
- Cell 2：预训练五大关键机制（数据->Token、标签右移、因果掩码、损失评估、训练稳定性）
- Cell 4：`使用设备: cuda`（或 cpu）

❓ 预判问题
- **Q：预训练数据需要标注吗？**
  A：不需要！这就是自监督学习的美妙之处。文本本身就包含标签——下一个字就是答案。所以才能用海量的互联网文本来训练。
- **Q：预训练和 Ch7 的关系是什么？**
  A：Ch7 是原理讲解，用独立的 TinyLlama 演示。CGT-02 是自建 GPT 流水线的第二步，复用 Part 1 组装好的 CustomGPT 模型。原理一样，工程上衔接更紧密。

➡️ 转场

> 环境就绪了。要训练模型，第一步是准备粮食——语料数据。

---

## 第二段：数据准备——语料加载与扩展（Cell 5-6）

📍 Cell 范围：Cell 5（Markdown 说明）、Cell 6（语料加载代码）

⏱ 时间分配：12 分钟

🎯 本段目标
- 理解预训练语料的来源与特点
- 理解语料扩展（重复）策略及其对小规模演示的必要性
- 了解实际应用中的语料规模差异

🗣 讲课话术

> 运行 Cell 6。（运行 Cell 6）看输出——**预训练语料 2000 条文本，总字符数 31,385**。
>
> 大家注意代码里的一个细节：语料不够 2000 条时会**重复扩展**。为什么？因为我们的演示语料很小，不重复的话训练步数不够，模型还没开始学就结束了。实际 LLM 的预训练数据是 TB 级别的，不需要重复。
>
> 打个比方：我们现在是给婴儿看一本薄薄的识字卡，反复翻，让它记住基本的字词搭配。而 GPT-4 的训练相当于读完了整个互联网。
>
> 另一个重要细节：代码在最后追加了**三条 ChatML 模板字符串**——`<|system|>`、`<|user|>`、`<|assistant|>` 这些特殊 token。为什么？因为后续 Part 3 SFT 要用 ChatML 格式做指令微调，如果分词器不认识这些特殊字符就会出问题。这叫**前瞻性设计**。
>
> Cell 5 提到实际应用中可以替换为 Wikipedia、Common Crawl、Books 等大规模语料。大家理解就好，我们的重点是走通流程。

👀 输出要点
- Cell 6：`预训练语料: 2000 条文本`
- Cell 6：`总字符数: 31385`
- 语料扩展策略：不足 2000 条时重复填充
- ChatML 种子文本追加：3 条模板字符串

❓ 预判问题
- **Q：重复语料会导致过拟合吗？**
  A：会。但在演示中这是有意为之——我们的目标是让模型快速学到基本模式，而非训出泛化能力强的模型。实际训练绝对要避免大量重复。
- **Q：为什么要在预训练阶段就加 ChatML 特殊 token？**
  A：分词器的词表在预训练阶段确定后就固定了。如果 SFT 时突然出现 `<|user|>` 这样的 token 但词表里没有，模型无法处理。所以要提前「预留席位」。
- **Q：31K 字符的数据量在实际中算什么级别？**
  A：极小。GPT-3 的训练数据约 3000 亿 token，LLaMA-2 约 2 万亿 token。我们的 31K 字符大约只有几千个 token，差了 8-9 个数量级。

➡️ 转场

> 语料有了，但模型不认识中文字符。下一步是用分词器把文字变成数字，然后构造训练样本。

---

## 第三段：数据集构建——分词器 + 滑动窗口 + DataLoader（Cell 7-10）

📍 Cell 范围：Cell 7（Markdown 标题）、Cell 8（PretrainDataset 类定义）、Cell 9（构建分词器）、Cell 10（创建数据集 + DataLoader）

⏱ 时间分配：15 分钟（分词器 5 分钟 + PretrainDataset 5 分钟 + DataLoader 5 分钟）

🎯 本段目标
- 理解字符级分词器的构建过程和词表大小
- 掌握 PretrainDataset 的滑动窗口机制（50% overlap）
- 理解 input_ids 和 labels 的右移关系
- 了解 train/val 划分和 DataLoader 的 batch 组织

🗣 讲课话术

> 先运行 Cell 9，构建分词器。（运行 Cell 9）看输出——**词表大小 381**。这是字符级分词器，每个汉字、标点、英文字母各占一个 token。相比 GPT-2 的 50K 词表，381 非常小，因为我们的语料只覆盖了这么多字符。
>
> 看编解码测试：「深度学习是人工智能的核心技术。」编码成 `[1, 248, 40, 107, 109, ...]`。注意开头的 `1` 是 BOS（Begin of Sequence）token。解码结果里有些字变成了 `<unk>`——说明个别字在词表中出现频率不够。这在实际中需要扩大语料来解决。
>
> 现在看 Cell 8 的 PretrainDataset。这是今天最关键的数据结构。它做了三件事：
>
> 第一，把所有文本编码后**拼接成一个超长的 ID 序列**。想象把 2000 条文本首尾相连，变成一条很长的数字链。
>
> 第二，用**滑动窗口**在这条长链上截取固定长度的训练样本。窗口大小是 `max_length=256`，步长是 `max_length//2=128`——也就是 **50% 的重叠**。为什么要重叠？打个比方：你读一本书，每次只看一页。如果每次翻一整页，页面边界处的句子就被切断了。但如果每次只翻半页，前一个样本的后半段就是下一个样本的前半段，上下文就连贯了。
>
> 第三，每个样本长度是 `max_length+1`，前 256 个 token 是 input_ids，后 256 个 token（右移一位）是 labels。这就是 **Teacher Forcing**——模型看到 token[0:255]，要预测 token[1:256]。
>
> 运行 Cell 10。（运行 Cell 10）**276 个训练样本**，90/10 划分后训练集 248、验证集 28。Batch size 16，每个 batch 的 shape 是 `[16, 256]`。
>
> 大家可以动手算一下：248 个样本 / 16 = 约 16 个 batch，也就是每个 epoch 跑 16 步。10 个 epoch 就是 160 步。

👀 输出要点
- Cell 9：`词表大小: 381`，字符级分词器
- Cell 9：编解码测试显示 BOS token（id=1）和部分 `<unk>`
- Cell 10：`创建了 276 个训练样本`，每个样本长度 256
- Cell 10：`训练集: 248 样本`，`验证集: 28 样本`
- Cell 10：`input_ids shape: torch.Size([16, 256])`，`labels shape: torch.Size([16, 256])`

❓ 预判问题
- **Q：50% overlap 会不会导致训练数据「水分太大」？**
  A：有一定数据膨胀，但每个样本的标签右移后预测的上下文不同，模型确实在学不同的语境。实际 LLM 通常不用 overlap，直接切分就够了，因为数据量足够大。
- **Q：max_length=256 和模型的 max_seq_len 要一致吗？**
  A：是的，必须一致。数据集的序列长度不能超过模型的位置编码范围。这里 GPTConfig 的 max_seq_len 也设为 256。
- **Q：为什么分词器需要保存和加载？**
  A：词表（字符到 ID 的映射）必须在后续 SFT、DPO、推理中保持一致。换了词表，模型就「听不懂」了。所以分词器保存到 `models/custom_gpt/tokenizer.pkl`。

➡️ 转场

> 数据管道搭好了。现在需要一个训练引擎——Trainer。它负责管理学习率、梯度裁剪、评估、保存 checkpoint。

---

## 休息 + 回顾（第 35-40 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 预训练的核心任务是 Next-Token Prediction——给定前文，预测下一个 token，本质是最小化交叉熵损失。
2. 我们的中文语料有 2000 条文本（31K 字符），字符级分词器的词表大小为 381。
3. PretrainDataset 用 50% overlap 的滑动窗口在长序列上截取 276 个训练样本，每个样本 256 tokens，input 和 label 右移一位对齐。

**下一段预告：** 我们要实现 PretrainTrainer，重点是 Warmup + Cosine Decay 学习率策略。

---

## 第四段：Trainer 实现——LR 调度 + 梯度裁剪 + 评估（Cell 11-12）

📍 Cell 范围：Cell 11（Markdown 标题）、Cell 12（PretrainTrainer 完整实现）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解 Warmup + Cosine Decay 学习率调度的原理
- 理解梯度裁剪（clip_grad_norm_）的作用
- 掌握 train-evaluate-save 的完整训练循环
- 理解困惑度（Perplexity）指标

🗣 讲课话术

> Cell 12 是今天代码量最大的一段，但逻辑很清晰。我们一块一块看。
>
> **第一块：学习率调度 `get_lr()`。** 这个函数实现了 Warmup + Cosine Decay。
>
> 打个比方：你刚开始学开车。前 50 步（warmup_steps=50）是在驾校低速练习，学习率从 0 线性增长到 3e-4。之后上路，速度最快。然后逐渐减速——Cosine 曲线是先慢后快地减速，最终趋近 0。
>
> 为什么要 Warmup？模型刚初始化时参数是随机的，梯度方向不稳定。如果一上来就用大学习率，就像新手开车猛踩油门，容易翻车（梯度爆炸）。Warmup 让优化器先「热身」。
>
> 为什么用 Cosine Decay？比线性衰减更平滑，训练后期可以精细调整参数。这是 GPT-3、LLaMA 等大模型的标准做法。
>
> **第二块：`train_epoch()` 训练循环。** 每个 batch 的流程是：
> 1. 更新学习率
> 2. 前向传播——模型输出 logits，和 labels 计算交叉熵 loss
> 3. 反向传播——`loss.backward()` 计算梯度
> 4. **梯度裁剪**——`clip_grad_norm_(model.parameters(), 1.0)`，把梯度的 L2 范数限制在 1.0 以内。就像给车装了限速器，防止单步更新太大导致训练崩溃
> 5. 优化器更新——`optimizer.step()`
>
> **第三块：`evaluate()` 验证评估。** 在验证集上计算 loss，然后算 **Perplexity = exp(loss)**。困惑度的直觉含义：模型平均在多少个选项中犹豫。PPL=10 意味着模型对每个 token 大约在 10 个候选中不确定。PPL=1 是完美预测。随机猜测的 PPL 等于词表大小 381。
>
> **第四块：`train()` 主函数。** 串联起训练、评估、保存最佳模型。每个 epoch 结束后，如果验证 loss 创新低，就保存 checkpoint。
>
> 注意优化器配置：`AdamW`，`betas=(0.9, 0.95)`，`weight_decay=0.01`。这些是 LLM 训练的标准超参数。

👀 输出要点
- Cell 12 无直接输出（纯类定义），重点理解代码结构
- 关键超参数：lr=3e-4, warmup_steps=50, max_grad_norm=1.0, weight_decay=0.01
- 学习率调度：Linear Warmup（前 50 步）+ Cosine Decay（之后）
- 评估指标：Loss 和 Perplexity（PPL = exp(loss)）

❓ 预判问题
- **Q：AdamW 和 Adam 有什么区别？**
  A：AdamW 把 weight decay 从梯度更新中解耦出来，直接对参数做衰减。在 Transformer 训练中效果更好，是当前 LLM 的标准选择。
- **Q：betas=(0.9, 0.95) 是什么意思？**
  A：beta1=0.9 控制一阶矩（梯度均值）的衰减，beta2=0.95 控制二阶矩（梯度方差）的衰减。GPT-3 论文用的就是 (0.9, 0.95)，比 PyTorch 默认的 (0.9, 0.999) 更激进，训练更不容易卡在平坦区。
- **Q：梯度裁剪的阈值 1.0 是怎么选的？**
  A：经验值。太小会限制学习速度，太大等于没裁剪。1.0 是 LLM 训练中最常用的值。

➡️ 转场

> Trainer 写好了，万事俱备。下面开炉炼丹！

---

## 第五段：执行预训练 + 可视化分析（Cell 13-17）

📍 Cell 范围：Cell 13（Markdown 标题）、Cell 14（模型配置）、Cell 15（创建模型 + Trainer）、Cell 16（执行预训练）、Cell 17（可视化训练过程）

⏱ 时间分配：15 分钟（配置 3 分钟 + 训练 7 分钟 + 可视化 5 分钟）

🎯 本段目标
- 理解模型配置参数与参数量的关系
- 观察 loss 从高到低的完整下降过程
- 理解困惑度变化的含义
- 通过可视化分析学习率调度的实际效果

🗣 讲课话术

> 运行 Cell 14 看模型配置。（运行 Cell 14）
>
> - `vocab_size: 381`——词表大小
> - `d_model: 384`——隐藏层维度
> - `n_layers: 6`——6 层 Transformer
> - `n_heads: 6`——6 个注意力头
> - `d_ff: 1536`——FFN 中间维度（4 倍 d_model）
> - 预估参数量 **~10.87M**
>
> 运行 Cell 15——实际参数量 **14.31M**。比 Part 1 的模型大一些，因为我们这次用了 6 层 384 维。
>
> 好，运行 Cell 16 开始训练！（运行 Cell 16）
>
> 大家看训练日志：
> - **Epoch 1：训练 Loss 5.5799，验证 Loss 4.9570，困惑度 142.16**。初始 loss 约等于 ln(381)=5.94，验证了理论值。困惑度 142 意味着模型在 142 个字符里犹豫。
> - **Epoch 3：训练 Loss 2.7883，验证 Loss 1.8164，困惑度 6.15**。只用了 3 个 epoch，模型就从 142 个里猜一个进步到 6 个里猜一个！
> - **Epoch 5：验证 Loss 0.7971，困惑度 2.22**。基本上在 2 个候选中选。
> - **Epoch 10：验证 Loss 0.2501，困惑度 1.28**。几乎能准确预测下一个字了。
>
> 注意每个 epoch 都在保存最佳模型——`保存最佳模型 (val_loss: ...)` 的提示一直在出现，说明验证 loss 持续下降，没有过拟合。为什么 Ch7 严重过拟合而这里没有？因为 Ch7 训了 2000 步但语料只有 4K 字符；我们这里虽然语料也不大（31K 字符），但只训 10 个 epoch（约 160 步），训练量相对克制。
>
> 运行 Cell 17 看可视化。（运行 Cell 17）
>
> 三张图：
> - **左图：训练损失**——蓝色是原始 loss，红色是平滑曲线。从 5.5 一路降到 0.2 左右，下降非常顺利。
> - **中图：验证损失**——绿色折线，10 个点对应 10 个 epoch，持续下降。
> - **右图：学习率调度**——看到了吗？前 50 步线性增长（Warmup），然后是漂亮的 Cosine 曲线逐渐归零。和我们之前讲的完全一致。

👀 输出要点
- Cell 14：`vocab_size: 381`, `d_model: 384`, `n_layers: 6`, `n_heads: 6`, `预估参数量: ~10.87M`
- Cell 15：`实际参数量: 14.31M`
- Cell 16 训练日志：
  - Epoch 1: Train Loss=5.5799, Val Loss=4.9570, PPL=142.16
  - Epoch 3: Train Loss=2.7883, Val Loss=1.8164, PPL=6.15
  - Epoch 5: Train Loss=1.2053, Val Loss=0.7971, PPL=2.22
  - Epoch 10: Train Loss=0.2713, Val Loss=0.2501, PPL=1.28
  - 每个 epoch 都保存了最佳模型
- Cell 17：三张可视化图（训练 Loss / 验证 Loss / 学习率调度）

❓ 预判问题
- **Q：PPL 降到 1.28 是不是太低了？是不是过拟合了？**
  A：对于 31K 字符的小语料来说，PPL 这么低确实说明模型基本「背住」了训练数据。但注意验证 loss 也在持续下降，说明泛化能力也在提升。真正的过拟合信号是验证 loss 上升而训练 loss 下降。
- **Q：14.31M 参数对 31K 字符是不是太大了？**
  A：是的，参数量远大于数据量，容量严重过剩。但我们的目的是走通流程、验证架构，不追求最优泛化。实际应用中应遵循 Chinchilla 法则：最优 token 数约等于参数量的 20 倍。
- **Q：为什么预估参数量（10.87M）和实际参数量（14.31M）不一样？**
  A：预估是 config 基于 d_model、n_layers 等做的粗算，没有计入 LayerNorm、bias、位置编码等「零碎」参数。实际参数量是精确统计所有可训练参数。

➡️ 转场

> 训练完了，loss 降了 20 多倍。但 loss 只是数字，我们要看模型到底学会了什么。

---

## 休息 + 回顾（第 70-75 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. PretrainTrainer 实现了完整的训练循环：Warmup + Cosine Decay 学习率、梯度裁剪（max_norm=1.0）、AdamW 优化器（betas=0.9/0.95）。
2. 14.31M 参数的模型训练 10 个 epoch，验证 Loss 从 4.96 持续降到 0.25，困惑度从 142 降到 1.28——模型从「381 个字里瞎猜」进步到「几乎能准确预测」。
3. 可视化显示 Warmup 前 50 步线性增长、之后 Cosine 平滑衰减，Loss 曲线下降顺利无振荡。

**下一段预告：** 让我们看看模型到底能生成什么样的文本，还要做温度实验和困惑度评估。

---

## 第六段：模型测试——生成 + 温度实验 + 困惑度（Cell 18-24）

📍 Cell 范围：Cell 18（Markdown 标题）、Cell 19（加载最佳模型）、Cell 20（generate_text 函数）、Cell 21（文本生成测试）、Cell 22（温度对比实验）、Cell 23（Markdown 标题）、Cell 24（困惑度计算）

⏱ 时间分配：10 分钟

🎯 本段目标
- 直观感受模型的生成能力和局限性
- 理解温度参数对生成多样性的影响
- 掌握困惑度评估方法及其含义

🗣 讲课话术

> 运行 Cell 19，加载最佳预训练模型。（运行 Cell 19）**14.31M 参数**加载完毕。
>
> Cell 20 定义了 `generate_text` 函数——给一个 prompt，模型一个 token 一个 token 地生成后续内容。使用 `temperature=0.8` 和 `top_k=50`。
>
> 运行 Cell 21 测试生成。（运行 Cell 21）四个 prompt 的结果：
>
> - 「深度学习」-> 「习习习习习长习习……」——大量重复！
> - 「机器学习是」-> 「是模阵是量量量……」——片段化的碎片
> - 「Transformer架构」-> 「机序规可释……用注力制理列」——能看到「注力」（注意力）、「序列」这样的碎片
> - 「语言模型」-> 「练要注解性」——很短
>
> 老实说，生成质量不高对吧？但大家要注意两点：第一，这些碎片里有真实的领域词汇——「注意力」「模型」「参数」「训练」；第二，字符级编码导致序列很长，模型很难学到长距离的语法结构。实际 LLM 用 BPE 分词 + 几十亿参数 + 万亿 token 数据才能生成流畅文本。
>
> 运行 Cell 22 看**温度实验**。（运行 Cell 22）Prompt 是「机器学习」：
> - **T=0.3**：「习习习习习习习……的的长长长长长律律」——极度保守，疯狂重复最高概率的字
> - **T=0.7**：「型练小让型练要立度回策。型练要注解性」——稍好一些，但仍有重复
> - **T=1.0**：「列文/ 步。拟tzU一能合阵分……逻」——开始引入低概率字符，更混乱
> - **T=1.5**：「号案」——高温度下概率太分散，生成了 EOS 就停了
>
> 温度的本质：**softmax(logits/T)**。T 越小，概率分布越尖锐，总选概率最高的字，确定性高但容易重复；T 越大，概率分布越平坦，低概率字也有机会被选，多样性高但质量差。
>
> 最后运行 Cell 24 计算困惑度。（运行 Cell 24）**验证 Loss: 0.2422，困惑度 1.27**。模型对验证集几乎是确定性预测。对比：随机猜测的困惑度是 381（词表大小），我们的模型好了 300 倍。
>
> 但要清醒地认识到：PPL 低不等于生成质量高。模型「背住」了训练数据中的模式，但缺乏泛化能力。这就是为什么需要大数据、大模型。

👀 输出要点
- Cell 19：`加载预训练模型: 14.31M`
- Cell 21：四个 prompt 的生成结果——有领域词汇碎片但不通顺，大量重复
- Cell 22：温度对比——T=0.3 重复、T=0.7 略好、T=1.0 混乱、T=1.5 过短
- Cell 24：`验证Loss: 0.2422`，`困惑度: 1.27`，对比随机猜测 PPL=381

❓ 预判问题
- **Q：为什么生成的文本这么差？Loss 不是很低了吗？**
  A：Loss 低是在「给定真实前文」的条件下预测准确。但生成时是自回归——前面生成的错误会累积放大（error propagation）。加上语料小、词表小，生成质量就很有限。
- **Q：实际产品中温度一般设多少？**
  A：对话场景 0.7-0.9，代码生成 0.2-0.5，创意写作 0.9-1.2。通常配合 top-k 或 top-p 一起用。
- **Q：PPL=1.27 在实际中算好吗？**
  A：不能跨数据集比较 PPL。我们的 PPL 低是因为训练集小且有重复。GPT-2 在 WikiText-103 上 PPL 约 20-30，但那是在更大、更多样的数据集上。
- **Q：预训练模型不能遵循指令，后续怎么办？**
  A：这正是 Part 3（SFT）要解决的问题。SFT 使用 (指令, 回答) 对来微调，让模型学会「听话」。当前模型只会续写，不会对话。

➡️ 转场

> 预训练阶段的所有实验都做完了。让我们回顾一下今天走过的完整流程。

---

## 第七段：总结 + 下一步展望（Cell 25）

📍 Cell 范围：Cell 25（总结 Markdown）

⏱ 时间分配：5 分钟

🎯 本段目标
- 回顾预训练全流程
- 明确预训练模型的能力与不足
- 建立对 SFT 的期待

🗣 讲课话术

> 浏览 Cell 25。今天我们完成了四件大事：
>
> **第一，数据准备**——加载 2000 条中文语料（31K 字符），用字符级分词器（vocab_size=381）编码，通过 50% overlap 滑动窗口切出 276 个训练样本。
>
> **第二，Trainer 实现**——Next-Token Prediction 训练循环，配备 Warmup + Cosine Decay 学习率、梯度裁剪、验证集评估、最佳模型保存。
>
> **第三，执行训练**——14.31M 参数模型训练 10 个 epoch，Loss 从 5.58 降到 0.25，PPL 从 142 降到 1.28。
>
> **第四，评估测试**——生成测试看到了领域词汇碎片但语法不通顺；温度实验展示了采样多样性的权衡；困惑度 1.27 远优于随机猜测的 381。
>
> 预训练后的模型学会了**基本的语言模式**——词语搭配、标点使用、一些领域术语。但它还**不会遵循指令**——你问它问题，它只会续写，不会回答。这就像一个会说话但不会对话的孩子。
>
> 下一步 Part 3 SFT，我们会用 ChatML 格式的 (指令, 回答) 数据来微调，让模型从「会说话」变成「会听话」。敬请期待！

👀 输出要点
- Cell 25 总结了四大模块：数据准备、Trainer、训练执行、模型评估
- 预训练模型的能力：基本语言模式、领域知识碎片、文本续写
- 预训练模型的不足：不能遵循指令、不能生成高质量回答
- 下一步：Part 3 SFT 训练（03_SFT_Training.ipynb）

❓ 预判问题
- **Q：如果想提高预训练质量，最该改什么？**
  A：按优先级——（1）增大语料，这是最根本的；（2）用 BPE 代替字符级编码，提高信息密度；（3）增大模型，但要和数据量匹配。
- **Q：SFT 会改变模型的所有参数吗？**
  A：标准 SFT 是全参数微调（full fine-tuning），所有参数都会更新。也可以用 LoRA 只更新低秩适配器，Part 4 会覆盖相关内容。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场 + 预训练概览 + 环境准备 | 0-4 |
| 8 | 数据准备：语料加载与扩展 | 5-6 |
| 20 | 数据集构建：分词器 + 滑动窗口 + DataLoader | 7-10 |
| 35 | **休息** | -- |
| 40 | Trainer 实现：LR 调度 + 梯度裁剪 + 评估 | 11-12 |
| 55 | 执行预训练 + 可视化分析 | 13-17 |
| 70 | **休息** | -- |
| 75 | 模型测试：生成 + 温度实验 + 困惑度 | 18-24 |
| 85 | 总结 + 下一步展望 | 25 |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)$$

$$\text{Perplexity} = e^{\mathcal{L}}$$

### 数据规模速查

| 指标 | 值 |
|:---|:---|
| 语料条数 | 2,000（含扩展重复） |
| 总字符数 | 31,385 |
| 词表大小（vocab_size） | 381 |
| 分词器类型 | 字符级（char） |
| max_length（序列长度） | 256 |
| 滑动窗口步长 | 128（50% overlap） |
| 训练样本数 | 276 -> 248（train）/ 28（val） |
| batch_size | 16 |
| tokens/batch | 4,096（16 x 256） |

### 模型配置速查

| 参数 | 值 |
|:---|:---|
| d_model | 384 |
| n_heads | 6 |
| n_layers | 6 |
| d_ff | 1,536（4 x d_model） |
| dropout | 0.1 |
| 预估参数量 | ~10.87M |
| 实际参数量 | 14.31M |

### 训练关键数值

| 指标 | 值 |
|:---|:---|
| 理论初始 Loss | ln(381) ≈ 5.94 |
| Epoch 1 Train/Val Loss | 5.5799 / 4.9570 |
| Epoch 1 PPL | 142.16 |
| Epoch 5 Train/Val Loss | 1.2053 / 0.7971 |
| Epoch 5 PPL | 2.22 |
| Epoch 10 Train/Val Loss | 0.2713 / 0.2501 |
| Epoch 10 PPL | 1.28 |
| 最终验证困惑度 | 1.27（compute_perplexity） |
| 随机猜测 PPL | 381（= vocab_size） |
| 学习率 | 3e-4 |
| Warmup 步数 | 50 |
| 总 Epoch 数 | 10 |
| 训练设备 | CUDA（GPU） |

---

## 附录 C：应急预案

### 场景 1：custom_gpt 模块导入失败

**症状：** Cell 4 报错 `ModuleNotFoundError: No module named 'custom_gpt'`

**应对：**
1. 确认已完成 Part 1（01_Model_Assembly.ipynb）并生成了 `custom_gpt.py`
2. 检查 `Custom_GPT_Training/` 目录下是否有 `custom_gpt.py`
3. 手动设置路径：`sys.path.insert(0, '/path/to/Custom_GPT_Training')`

### 场景 2：语料文件找不到

**症状：** Cell 6 使用内置示例文本而非外部语料

**应对：**
1. 检查 `data/custom_pretrain_corpus.txt` 是否存在
2. 确认 `resolve_data_dir()` 能找到正确的 data 目录
3. 内置示例文本也能演示流程，只是语料质量较低

### 场景 3：训练时间过长

**症状：** CPU 上训练超过 10 分钟

**应对：**
1. 将 EPOCHS 从 10 减少到 3-5
2. 将 BATCH_SIZE 从 16 增大到 32（减少步数）
3. 核心概念不受影响——仍能观察到 Loss 下降趋势

### 场景 4：CUDA 内存不足

**症状：** `RuntimeError: CUDA out of memory`

**应对：**
1. 减小 BATCH_SIZE（从 16 改为 8）
2. 减小 MAX_LENGTH（从 256 改为 128）
3. 切换到 CPU：手动设置 `device = 'cpu'`

### 场景 5：分词器编解码不一致

**症状：** 编码后解码出现 `<unk>` 或缺字

**应对：**
1. 这是字符级分词器的正常现象——低频字可能未进入词表
2. 可通过降低 `min_freq=1` 或增大语料来覆盖更多字符
3. 不影响训练流程，只是生成质量略有损失

### 场景 6：生成文本全是重复字符

**症状：** 模型输出类似「习习习习习习」的死循环

**应对：**
1. 这是小模型 + 小数据的常见现象，不是 bug
2. 可尝试提高温度（temperature=1.0）增加多样性
3. 也可降低 top_k 值（如 top_k=10）限制候选范围
4. 根本解决需要增大模型和数据规模